In [ ]:
!pip install openai faiss-cpu pypdf

In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"]= getpass("Enter your API key: ")
#from dotenv import load_dotenv
#load_dotenv(".env")
#api_key = os.getenv("OPENAI_API_KEY")


Enter your API key: ··········


In [ ]:
from google.colab import files
uploaded= files.upload()

Saving AITools_Unit-1.pdf to AITools_Unit-1 (1).pdf


In [ ]:
from pypdf import PdfReader

def read_pdf(file):
  reader = PdfReader(file)
  text = ""
  for page in reader.pages:
    text += page.extract_text()
  return text

file_name = list(uploaded.keys())[0]
text = read_pdf(file_name)
print(text[:400])

UNIT-1 
Introduction to Artificial Intelligence: What is AI, Foundations of AI, Goals of AI, and Applications of 
AI. 
 
Q) Define AI. Describe the organization of AI Definition. 
John McCarthy in mid -1950’scoined the term ― Artificial Intelligence ‖ which he 
would define as ―the science and engineering of making intelligent machines‖ 
AI is about teaching the machines to learn, to act, and t hi


In [ ]:
def split_text(text, chunk_size=200):
  chunks = []
  for i in range(0 ,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])
  return chunks
chunks = split_text(text)
print(len(chunks))

153


In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
def get_embedding(text):
  return client.embeddings.create(
      model = "text-embedding-3-small",
      input = text
  ).data[0].embedding

embeddings = [get_embedding(chunk) for chunk in chunks]

In [ ]:
import faiss
import numpy as np
dimension=len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("Stored in faiss")

Stored in faiss


In [ ]:
def search(query,k=2):

  query_embedding = np.array([get_embedding(query)]).astype("float32")

  distances,indices = index.search(query_embedding , k)

  results = [chunks[i] for i in indices[0]]
  return results

In [ ]:
def ask_question(query):
  relevant_chunks = search(query)

  context = "\n".join(relevant_chunks)
  prompt = f"""
  Answer the question based on the context mentioned in the pdf, if the question is out of context say "I don't know."

  Context:
  {context}

  Question:
  {query}
  """
  response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages= [{"role":"user","content":prompt}]
  )
  return response.choices[0].message.content

In [ ]:
question = "what is AI?"
answer=ask_question(question)
print(answer)

AI, or Artificial Intelligence, is defined as "the science and engineering of making intelligent machines." It involves teaching machines to learn, act, and think intelligently.
